# PSA-MT — Notebook 2: mT5-small Transfer Learning

**Purpose:** train mT5 independently from the NLLB notebook while reusing the exact shared train/dev/test split.

This notebook covers:
- mT5-small pretrained checkpoint persistence,
- six translation directions,
- few-shot cap,
- Ekegusii-focused word-dropout augmentation,
- per-direction mT5 baseline,
- combined multilingual mT5 (core transfer-learning model),
- freeze-vs-full-fine-tuning ablation,
- checkpoint after every epoch,
- per-epoch metrics,
- final BLEU / SacreBLEU / chrF++ / COMET,
- deployment-ready stable `best/` checkpoints.

**Run this notebook after Notebook 1 has created `shared_artifacts/`.**

## 1. Install dependencies

In [190]:
#!pip -q install -U transformers datasets accelerate sentencepiece sacrebleu mlflow evaluate

## 1b. All imports (everything below this cell only *uses* these -- no more scattered imports)

In [191]:
from pathlib import Path
import os, json, time, random, math, shutil, subprocess, sys, inspect

import numpy as np
import pandas as pd
import torch
import sacrebleu
import mlflow

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback,
)
from transformers.trainer_utils import get_last_checkpoint

# NOTE: `comet` is deliberately NOT imported here -- it's installed and imported in its
# own isolated cell later, only once you actually reach the COMET section. Importing it
# here would fail before that install ever runs.

print("All core imports loaded.")

All core imports loaded.


In [192]:
# ==========================
# Suppress warnings & logs
# Run this as the FIRST cell
# ==========================

import os
import warnings
import logging

# Silence Python warnings
warnings.filterwarnings("ignore")

# Silence Hugging Face warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Silence MLflow Git warnings
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

# Reduce TensorFlow logs (harmless if TensorFlow isn't used)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Silence Hugging Face Hub progress/warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Silence Python logging
logging.getLogger().setLevel(logging.ERROR)

# Silence specific libraries
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

print("✅ Warnings and logs suppressed.")

✅ Warnings and logs suppressed.


In [193]:
# =========================
# PSA-MT shared configuration
# =========================
# Check Colab's own internal marker file directly -- more reliable than "did the import
# succeed", since a google.colab-like package can exist without being the real hosted VM.
IN_COLAB = os.path.exists("/var/colab/hostname")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    INPUT_DIR = Path("/content/drive/MyDrive/NLP_Translation")
else:
    INPUT_DIR = Path.cwd()

# ---- ALL pipeline outputs go into ONE new subfolder inside your existing folder ----
# Nothing else already in NLP_Translation (your other csvs/notebooks) is touched or moved.
OUTPUT_ROOT = INPUT_DIR / "PSA-MT-Outputs"

SHARED_DIR = OUTPUT_ROOT / "shared_artifacts"
DATA_DIR = SHARED_DIR / "data_processed"
MODELS_DIR = OUTPUT_ROOT / "models"
BASE_MODELS_DIR = MODELS_DIR / "base_pretrained"
FT_MODELS_DIR = MODELS_DIR / "fine_tuned"
RESULTS_DIR = OUTPUT_ROOT / "results"
LOGS_DIR = OUTPUT_ROOT / "logs"
MLFLOW_DIR = OUTPUT_ROOT / "mlflow"
for p in [SHARED_DIR, DATA_DIR, BASE_MODELS_DIR, FT_MODELS_DIR, RESULTS_DIR, LOGS_DIR, MLFLOW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Shared experimental policy
FEWSHOT_N = 2000
MAX_LEN = 128
EPOCHS = 5              # was 4 -- consistent with NLLB's tuning (chrF was still climbing at epoch 4)
TRAIN_BATCH = 32         # was 8 -- verified ~14x throughput on this A100, plenty of VRAM headroom
EVAL_BATCH = 32          # was 8 -- matched to TRAIN_BATCH
GRAD_ACCUM = 2
FREEZE_ENCODER = True
# Partial-layer freezing (criteria differs by model -- each has a different total encoder
# depth): mT5-small has 8 encoder layers, freeze the bottom 4. NLLB-200-distilled-600M has
# 12, freeze the bottom 6. None -> fall back to whole-encoder freeze via FREEZE_ENCODER.
MT5_FREEZE_LAYERS = 4    # out of 8 total
NLLB_FREEZE_LAYERS = 6   # out of 12 total (set in the NLLB notebook's own config)
USE_GRADIENT_CHECKPOINTING = True
# Drive space is limited (we hit the quota once already) -- keep only 2 checkpoints per
# run instead of all EPOCHS. Note: with load_best_model_at_end=True, HF Trainer protects
# the single BEST checkpoint from deletion even if it isn't one of the most recent -- so
# this really means "keep the best, plus 1 more recent one," not strictly "last 2 by time."
CHECKPOINT_KEEP = 2
USE_AUGMENTATION = True
AUGMENT_FACTOR = 2
AUGMENT_DROP_PROB = 0.10
USE_COMET = True

# FIXED: was False -- this is why mT5 was still capping at FEWSHOT_N=2000 instead of
# training on the full dataset, as agreed. Now matches the decision already applied to NLLB.
FULL_DATASET_RUN = True

# No longer needs manual setting -- both training functions now auto-detect and
# resume from each run's own last checkpoint. Left here only for a manual override
# if you ever want to force a specific checkpoint path.
RESUME_CHECKPOINT = None

print("INPUT folder  (read-only -- your existing files):", INPUT_DIR)
print("OUTPUT folder (everything this pipeline creates):", OUTPUT_ROOT)
print("Epochs:", EPOCHS)
print("Train/eval batch size:", TRAIN_BATCH)
print("Full dataset run:", FULL_DATASET_RUN)

INPUT folder  (read-only -- your existing files): /home/jovyan/PSA_MT_Project
OUTPUT folder (everything this pipeline creates): /home/jovyan/PSA_MT_Project/PSA-MT-Outputs
Epochs: 5
Train/eval batch size: 32
Full dataset run: True


## 2. Hardware check

In [194]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
if DEVICE=="cpu":
    print("WARNING: mT5 training on CPU may take many hours.")

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM GB: 79.25


## 3. Experiment tracking (MLflow)

Satisfies "set up experiment tracking (W&B or MLflow)." Every training run below -- per-direction,
combined, and both ablations -- logs its hyperparameters and metrics here.

In [195]:
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DIR}/mlflow.db")
mlflow.set_experiment("psa-mt-mt5")
print("MLflow tracking to:", f"sqlite:///{MLFLOW_DIR}/mlflow.db")
print("View later with: mlflow ui --backend-store-uri", f"sqlite:///{MLFLOW_DIR}/mlflow.db")

MLflow tracking to: sqlite:////home/jovyan/PSA_MT_Project/PSA-MT-Outputs/mlflow/mlflow.db
View later with: mlflow ui --backend-store-uri sqlite:////home/jovyan/PSA_MT_Project/PSA-MT-Outputs/mlflow/mlflow.db


## 4. Load the exact shared split

**No new random split is created here.** This is essential for a fair comparison with NLLB.

In [196]:
required=DATA_DIR/"English_to_Kiswahili.train.csv"
if not required.exists():
    raise FileNotFoundError("Shared artifacts not found. Run Notebook 1 first.")
manifest=json.loads((SHARED_DIR/"split_manifest.json").read_text())
print(json.dumps(manifest,indent=2))

DIRECTIONS=[
    "English_to_Kiswahili","Kiswahili_to_English",
    "English_to_Ekegusii","Ekegusii_to_English",
    "Kiswahili_to_Ekegusii","Ekegusii_to_Kiswahili"
]
for slug in DIRECTIONS:
    for sp in ["train","dev","test"]:
        assert (DATA_DIR/f"{slug}.{sp}.csv").exists()
print("Shared train/dev/test artifacts verified.")

{
  "seed": 42,
  "ratios": {
    "train": 0.8,
    "dev": 0.1,
    "test": 0.1
  },
  "source_file": "/home/jovyan/PSA_MT_Project/kenyan_psa_multilingual_dataset.csv",
  "created_at": "2026-08-01 21:19:16"
}
Shared train/dev/test artifacts verified.


## 5. mT5 configuration

**Epochs = 4.**

We save a checkpoint at the end of every epoch. mT5 is kept in fp32 here for stability; if the selected GPU supports bf16 and you have tested it, that can be enabled later.

The six per-direction models are useful as a baseline. The combined model is the main transfer-learning experiment.

In [197]:

MT5_NAME="google/mt5-small"
BASE_MT5=BASE_MODELS_DIR/"mt5-small"
MT5_OUTPUT=FT_MODELS_DIR/"mt5"
# HF cache lives in LOCAL Colab storage, not Drive -- it's just re-downloadable public
# model files (unlike your fine-tuned checkpoints, which DO need to be on Drive).
# This is what avoids repeating the Drive-quota problem from COMET's checkpoint download.
_hf_cache_root = Path("/content/hf_cache") if IN_COLAB else (OUTPUT_ROOT/"hf_cache")
os.environ["HF_HOME"]=str(_hf_cache_root)
os.environ["TRANSFORMERS_CACHE"]=str(_hf_cache_root/"transformers")
os.makedirs(os.environ["TRANSFORMERS_CACHE"],exist_ok=True)
print("Base:",BASE_MT5)
print("Fine-tuned:",MT5_OUTPUT)

Base: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/base_pretrained/mt5-small
Fine-tuned: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/mt5


## 6. Download once and persist the pretrained mT5 checkpoint

In [198]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

def load_or_save_base_mt5():
    BASE_MT5.mkdir(parents=True, exist_ok=True)

    try:
        tok = AutoTokenizer.from_pretrained(BASE_MT5, use_fast=False)
        model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MT5)
        print("Loaded mT5 pretrained checkpoint from Drive.")
    except Exception as e:
        print(f"Local checkpoint could not be loaded:\n{e}")
        print("Downloading a fresh mT5 checkpoint...")

        tok = AutoTokenizer.from_pretrained(MT5_NAME, use_fast=False)
        model = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME)

        tok.save_pretrained(BASE_MT5)
        model.save_pretrained(BASE_MT5)

        print("Downloaded and saved fresh mT5 checkpoint.")

    return tok, model

tok0, model0 = load_or_save_base_mt5()
del model0
if torch.cuda.is_available(): torch.cuda.empty_cache()

Loaded mT5 pretrained checkpoint from Drive.


## 7. Dataset loading + low-resource augmentation

In [199]:
def word_dropout_noise(text,drop_prob=AUGMENT_DROP_PROB,rng=None):
    rng=rng or random
    words=str(text).split()
    kept=[w for w in words if rng.random()>drop_prob]
    return " ".join(kept) if kept else str(text)

def load_direction(slug,fewshot="default",augment=True):
    # fewshot="default" -> FEWSHOT_N unless FULL_DATASET_RUN=True (then all rows).
    # fewshot=None -> explicitly all rows, regardless of FULL_DATASET_RUN.
    if fewshot=="default":
        fewshot=None if FULL_DATASET_RUN else FEWSHOT_N
    out={}
    for sp in ["train","dev","test"]:
        df=pd.read_csv(DATA_DIR/f"{slug}.{sp}.csv").dropna(subset=["src_text","tgt_text"])
        if sp=="train":
            if fewshot is not None:
                df=df.sample(min(fewshot,len(df)),random_state=SEED).reset_index(drop=True)
            else:
                df=df.sample(frac=1,random_state=SEED).reset_index(drop=True)  # shuffle, keep all rows
            if augment and USE_AUGMENTATION and "Ekegusii" in slug:
                extra=df.copy()
                rng=random.Random(SEED)
                extra["src_text"]=extra["src_text"].apply(lambda x:word_dropout_noise(x,rng=rng))
                df=pd.concat([df,extra],ignore_index=True)
        out[sp]=Dataset.from_pandas(df,preserve_index=False)
    return DatasetDict(out)

def freeze_encoder(model, num_layers=None):
    """num_layers=None -> freeze the WHOLE encoder (original behavior).
    num_layers=N -> freeze only the bottom N encoder layers, leave the rest trainable
    (the partial-freezing technique -- a middle ground between full-freeze and no-freeze)."""
    encoder = model.get_encoder()
    if num_layers is None:
        for p in encoder.parameters():
            p.requires_grad = False
        return
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers:
            for p in layer.parameters():
                p.requires_grad = False

def preprocess_mt5(tokenizer,src,tgt):
    def fn(batch):
        prompt=[f"translate {src} to {tgt}: {x}" for x in batch["src_text"]]
        enc=tokenizer(prompt,max_length=MAX_LEN,truncation=True)
        lab=tokenizer(text_target=batch["tgt_text"],max_length=MAX_LEN,truncation=True)
        enc["labels"]=lab["input_ids"]
        return enc
    return fn

## 8. Per-epoch checkpoint and metric persistence

In [68]:
from tqdm.auto import tqdm

class EpochArtifactCallback(TrainerCallback):
    """Captures BOTH per-step training logs and per-epoch eval metrics to disk --
    so nothing is lost if the container resets, and prints clean, readable lines
    instead of raw HF Trainer dicts. Two files land in run_dir:
      training_steps.csv  -- every logged step: step, epoch, loss, grad_norm, lr
      epoch_metrics.csv   -- every epoch-end eval: bleu, chrf, eval_loss, runtime, etc.
    """
    def __init__(self, run_dir):
        self.run_dir = Path(run_dir); self.run_dir.mkdir(parents=True, exist_ok=True)
        self.step_rows = []
        self.eval_rows = []
        self.step_log_path = self.run_dir / "training_steps.csv"
        self.epoch_log_path = self.run_dir / "epoch_metrics.csv"

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            row = {
                "step": state.global_step,
                "epoch": round(logs.get("epoch", state.epoch), 4),
                "loss": logs.get("loss"),
                "grad_norm": logs.get("grad_norm"),
                "learning_rate": logs.get("learning_rate"),
            }
            self.step_rows.append(row)
            pd.DataFrame(self.step_rows).to_csv(self.step_log_path, index=False)
            gn = row["grad_norm"]; lr = row["learning_rate"]
            if gn is not None:
                print(f"  step {row['step']:>5} | epoch {row['epoch']:>6.3f} | "
                      f"loss {row['loss']:>7.4f} | lr {lr:.2e} | grad_norm {gn:>6.3f}")
            else:
                print(f"  step {row['step']:>5} | epoch {row['epoch']:>6.3f} | loss {row['loss']:>7.4f} | lr {lr:.2e}")
        elif "eval_loss" in logs:
            row = {"step": state.global_step, "epoch": round(logs.get("epoch", state.epoch), 4), **logs}
            self.eval_rows.append(row)
            pd.DataFrame(self.eval_rows).to_csv(self.epoch_log_path, index=False)
            print(f"  >>> EPOCH {row['epoch']:.0f} EVAL  |  "
                  f"loss {logs.get('eval_loss', float('nan')):.4f}  |  "
                  f"bleu {logs.get('eval_bleu', float('nan')):.2f}  |  "
                  f"chrf {logs.get('eval_chrf', float('nan')):.2f}  |  "
                  f"runtime {logs.get('eval_runtime', float('nan')):.1f}s")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            row = {"epoch": state.epoch, "step": state.global_step, **metrics}
            (self.run_dir / f"epoch_{int(round(state.epoch)):02d}_metrics.json").write_text(
                json.dumps(row, indent=2, default=str))


class TqdmProgressCallback(TrainerCallback):
    """A visual progress bar, completely separate from EpochArtifactCallback's clean
    text logging -- this just tracks step-by-step progress visually, it doesn't print
    or save anything, so it can't duplicate or conflict with the other callback."""
    def on_train_begin(self, args, state, control, **kwargs):
        self.bar = tqdm(total=state.max_steps, desc="Training", unit="step")

    def on_step_end(self, args, state, control, **kwargs):
        self.bar.n = state.global_step
        self.bar.refresh()

    def on_train_end(self, args, state, control, **kwargs):
        self.bar.close()

def compute_metrics_builder(tok):
    def compute(eval_pred):
        preds,labels=eval_pred
        if isinstance(preds,tuple): preds=preds[0]
        preds=np.where(preds!=-100,preds,tok.pad_token_id)   # <-- ADD THIS LINE
        labels=np.where(labels!=-100,labels,tok.pad_token_id)
        p=tok.batch_decode(preds,skip_special_tokens=True)
        r=tok.batch_decode(labels,skip_special_tokens=True)
        return {"bleu":sacrebleu.corpus_bleu(p,[r]).score,
                "chrf":sacrebleu.corpus_chrf(p,[r],word_order=2).score}
    return compute

# ---- transformers version compatibility -------------------------------------
# Newer transformers releases renamed Trainer's `tokenizer=` kwarg to
# `processing_class=`. This picks whichever one the installed version wants,
# so training doesn't crash if `pip install -U transformers` pulled a newer release.

def build_seq2seq_trainer(model, args, train_ds, eval_ds, collator, compute_metrics,
                          tokenizer, callbacks=None):
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=collator, compute_metrics=compute_metrics)
    if callbacks:
        kw["callbacks"] = callbacks
    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    kw["processing_class" if "processing_class" in params else "tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kw)

## 9. Train one mT5 per direction

This gives the **per-direction baseline** for all six directions. Each model receives at most 2,000 clean examples, with an additional noised copy for Ekegusii-involving directions.

In [19]:
def train_mt5_direction(direction,freeze=True,augment=True,tag="per_direction"):
    run_dir=MT5_OUTPUT/tag/direction
    run_dir.mkdir(parents=True,exist_ok=True)

    # Safe to re-run this whole notebook after any disconnect: a direction that already
    # finished is skipped entirely (no wasted GPU time, no accidental overwrite).
    final_path=run_dir/"final_test_metrics.csv"
    if final_path.exists():
        print(f"  [{tag}/{direction}] already completed -- loading saved result, skipping retrain.")
        return pd.read_csv(final_path).iloc[0].to_dict()

    src,tgt=direction.split("_to_")
    dsd=load_direction(direction,augment=augment)  # respects FULL_DATASET_RUN via "default"
    tok=AutoTokenizer.from_pretrained(BASE_MT5)
    model=AutoModelForSeq2SeqLM.from_pretrained(BASE_MT5).to(DEVICE)
    if freeze: freeze_encoder(model, num_layers=MT5_FREEZE_LAYERS)
    enc=dsd.map(preprocess_mt5(tok,src,tgt),batched=True,remove_columns=dsd["train"].column_names)

    args=Seq2SeqTrainingArguments(
        output_dir=str(run_dir),learning_rate=1e-3,
        per_device_train_batch_size=TRAIN_BATCH,per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=EPOCHS,
        optim="adafactor",weight_decay=0.0,predict_with_generate=True,
        generation_max_length=MAX_LEN,fp16=False,
        eval_strategy="epoch",save_strategy="epoch",save_total_limit=CHECKPOINT_KEEP,
        load_best_model_at_end=True,metric_for_best_model="chrf",greater_is_better=True,
        logging_steps=25,report_to=["mlflow"],gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        disable_tqdm=True,
    )
    trainer=build_seq2seq_trainer(
        model=model,args=args,train_ds=enc["train"],eval_ds=enc["dev"],
        collator=DataCollatorForSeq2Seq(tok,model=model),
        compute_metrics=compute_metrics_builder(tok),tokenizer=tok,
        callbacks=[EpochArtifactCallback(run_dir), TqdmProgressCallback()]
    )
    from transformers.trainer_callback import PrinterCallback
    trainer.remove_callback(PrinterCallback)
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except ImportError:
        pass

    # Auto-detect THIS direction's own last checkpoint -- no manual RESUME_CHECKPOINT
    # setting needed. Falls back to the manual override only if you set one yourself.
    resume=RESUME_CHECKPOINT or get_last_checkpoint(str(run_dir))
    if resume:
        print(f"  [{tag}/{direction}] resuming automatically from: {resume}")
    with mlflow.start_run(run_name=f"mt5-{tag}-{direction}"):
        mlflow.log_params({"model":"mT5-small","direction":direction,"tag":tag,"epochs":EPOCHS,
                           "freeze_encoder":freeze,"freeze_layers":MT5_FREEZE_LAYERS if freeze else 0,
                           "learning_rate":1e-3,"train_batch":TRAIN_BATCH,"grad_accum":GRAD_ACCUM,
                           "full_dataset_run":FULL_DATASET_RUN})
        t0=time.time()
        trainer.train(resume_from_checkpoint=resume)
        runtime=(time.time()-t0)/60

        best=run_dir/"best"
        trainer.save_model(best); tok.save_pretrained(best)
        # Evaluate held-out test
        test=enc["test"]
        ev=trainer.predict(test,metric_key_prefix="test")
        final={"model":"mT5-small","setting":tag,"direction":direction,
               "bleu":ev.metrics.get("test_bleu"),"chrf":ev.metrics.get("test_chrf"),
               "runtime_min":runtime,"epochs":EPOCHS,"freeze_encoder":freeze,
               "full_dataset_run":FULL_DATASET_RUN}
        mlflow.log_metrics({k:v for k,v in final.items() if isinstance(v,(int,float))})

        # Save a handful of actual translations, same as NLLB, for the audience display
       # Save a handful of actual translations, same as NLLB, for the audience display
        preds_ids = ev.predictions[0] if isinstance(ev.predictions, tuple) else ev.predictions
        preds_ids = np.where(preds_ids != -100, preds_ids, tok.pad_token_id)   # <-- ADD THIS LINE
        labels = np.where(ev.label_ids != -100, ev.label_ids, tok.pad_token_id)
        decoded_preds = tok.batch_decode(preds_ids, skip_special_tokens=True)
        decoded_refs = tok.batch_decode(labels, skip_special_tokens=True)
        sample_df = pd.DataFrame({
        "source": dsd["test"]["src_text"][:10],
        "prediction": decoded_preds[:10],
        "reference": decoded_refs[:10],  })
        sample_df.to_csv(run_dir / "sample_translations.csv", index=False)

    pd.DataFrame([final]).to_csv(final_path,index=False)
    print(final)
    del model,trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final

# Train only the selected directions
DIRECTIONS = [
    "English_to_Ekegusii",
    "English_to_Kiswahili",
    "Ekegusii_to_Kiswahili",
    "Kiswahili_to_Ekegusii",   # NEW -- matches NLLB's exact scope for direct comparison
]

per_direction = []

for direction in DIRECTIONS:
    print(f"\n{'='*100}")
    print(f"Training: {direction}")
    print(f"{'='*100}")

    per_direction.append(
        train_mt5_direction(
            direction=direction,
            freeze=FREEZE_ENCODER,
            augment=True,
            tag="per_direction"
        )
    )

results_df = pd.DataFrame(per_direction)
display(results_df)

results_df.to_csv(
    RESULTS_DIR / "mt5_selected_directions_final.csv",
    index=False
)


Training: English_to_Ekegusii
  [per_direction/English_to_Ekegusii] already completed -- loading saved result, skipping retrain.

Training: English_to_Kiswahili
  [per_direction/English_to_Kiswahili] already completed -- loading saved result, skipping retrain.

Training: Ekegusii_to_Kiswahili
  [per_direction/Ekegusii_to_Kiswahili] already completed -- loading saved result, skipping retrain.

Training: Kiswahili_to_Ekegusii
  [per_direction/Kiswahili_to_Ekegusii] already completed -- loading saved result, skipping retrain.


,model,setting,direction,bleu,chrf,runtime_min,epochs,freeze_encoder,full_dataset_run
0,mT5-small,per_direction,English_to_Ekegusii,13.373422,36.135920,29.755314,5,True,True
1,mT5-small,per_direction,English_to_Kiswahili,77.990718,85.780823,0.026502,5,True,True
2,mT5-small,per_direction,Ekegusii_to_Kiswahili,64.332173,73.432127,39.891198,5,True,True
3,mT5-small,per_direction,Kiswahili_to_Ekegusii,13.466466,35.863393,50.138063,5,True,True


## **zero-shot baseline**

In [33]:
mt5_zero_path = RESULTS_DIR / "mt5_zero_shot.csv"
if mt5_zero_path.exists():
    print("mT5 zero-shot already computed -- loading saved result.")
    mt5_zero_results = pd.read_csv(mt5_zero_path).to_dict("records")
else:
    mt5_zero_results = []
    for direction in DIRECTIONS:
        src, tgt = direction.split("_to_")
        ds = load_direction(direction, fewshot=None, augment=False)["test"]
        tok = AutoTokenizer.from_pretrained(BASE_MT5)
        model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MT5).to(DEVICE)
        t0 = time.time()
        preds = generate_mt5(model, tok, list(ds["src_text"]), src, tgt)
        refs = list(ds["tgt_text"])
        bleu = sacrebleu.corpus_bleu(preds, [refs]).score
        chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
        metrics = {"model":"mT5-small","setting":"zero-shot","direction":direction,
                  "bleu":round(bleu,2),"chrf":round(chrf,2),"runtime_min":(time.time()-t0)/60}
        mt5_zero_results.append(metrics)
        print(metrics)
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    pd.DataFrame(mt5_zero_results).to_csv(mt5_zero_path, index=False)

display(pd.DataFrame(mt5_zero_results))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

{'model': 'mT5-small', 'setting': 'zero-shot', 'direction': 'English_to_Ekegusii', 'bleu': 0.0, 'chrf': 1.09, 'runtime_min': 0.3968025843302409}


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

{'model': 'mT5-small', 'setting': 'zero-shot', 'direction': 'English_to_Kiswahili', 'bleu': 0.0, 'chrf': 1.09, 'runtime_min': 0.39678940375645955}


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

{'model': 'mT5-small', 'setting': 'zero-shot', 'direction': 'Ekegusii_to_Kiswahili', 'bleu': 0.01, 'chrf': 1.24, 'runtime_min': 0.42465409835179646}


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

{'model': 'mT5-small', 'setting': 'zero-shot', 'direction': 'Kiswahili_to_Ekegusii', 'bleu': 0.01, 'chrf': 1.45, 'runtime_min': 0.4582248131434123}


,model,setting,direction,bleu,chrf,runtime_min
0,mT5-small,zero-shot,English_to_Ekegusii,0.00,1.09,0.396803
1,mT5-small,zero-shot,English_to_Kiswahili,0.00,1.09,0.396789
2,mT5-small,zero-shot,Ekegusii_to_Kiswahili,0.01,1.24,0.424654
3,mT5-small,zero-shot,Kiswahili_to_Ekegusii,0.01,1.45,0.458225


## 10. Train the combined multilingual mT5

This is the **core transfer-learning model**: one model learns all six directions jointly. Because the Ekegusii directions are augmented and each direction is capped equally, English↔Kiswahili does not simply dominate by dataset size.

The final checkpoint is evaluated separately on every direction's unchanged test set.

In [23]:
def load_combined():
    fewshot=None if FULL_DATASET_RUN else FEWSHOT_N
    parts={"train":[],"dev":[],"test":[]}
    for slug in DIRECTIONS:
        src,tgt=slug.split("_to_")
        for sp in ["train","dev","test"]:
            df=pd.read_csv(DATA_DIR/f"{slug}.{sp}.csv").dropna(subset=["src_text","tgt_text"]).copy()
            if sp=="train":
                if fewshot is not None:
                    df=df.sample(min(fewshot,len(df)),random_state=SEED).reset_index(drop=True)
                else:
                    df=df.sample(frac=1,random_state=SEED).reset_index(drop=True)
                if USE_AUGMENTATION and "Ekegusii" in slug:
                    extra=df.copy(); rng=random.Random(SEED)
                    extra["src_text"]=extra["src_text"].apply(lambda x:word_dropout_noise(x,rng=rng))
                    df=pd.concat([df,extra],ignore_index=True)
            df["src_lang"]=src; df["tgt_lang"]=tgt; df["direction"]=slug
            parts[sp].append(df)
    return {sp:pd.concat(parts[sp],ignore_index=True).sample(frac=1,random_state=SEED).reset_index(drop=True)
            for sp in parts}

def preprocess_combined(tok):
    def fn(batch):
        prompts=[f"translate {s} to {t}: {x}" for s,t,x in zip(batch["src_lang"],batch["tgt_lang"],batch["src_text"])]
        enc=tok(prompts,max_length=MAX_LEN,truncation=True)
        enc["labels"]=tok(text_target=batch["tgt_text"],max_length=MAX_LEN,truncation=True)["input_ids"]
        return enc
    return fn

def generate_mt5(model,tok,texts,src,tgt,batch=8):
    model.eval(); out=[]
    for i in range(0,len(texts),batch):
        batch_text=list(texts[i:i+batch])
        prompts=[f"translate {src} to {tgt}: {x}" for x in batch_text]
        encb=tok(prompts,return_tensors="pt",padding=True,truncation=True,max_length=MAX_LEN).to(model.device)
        gen=model.generate(**encb,max_length=MAX_LEN)
        out.extend(tok.batch_decode(gen,skip_special_tokens=True))
    return [x.strip() for x in out]

def train_mt5_combined():
    run_dir=MT5_OUTPUT/"combined"
    run_dir.mkdir(parents=True,exist_ok=True)
    final_path=RESULTS_DIR/"mt5_combined_final.csv"

    # Safe to re-run after any disconnect: skip entirely if this already finished.
    if final_path.exists():
        print("  [combined] already completed -- loading saved result, skipping retrain.")
        return pd.read_csv(final_path)

    combined=load_combined()
    tok=AutoTokenizer.from_pretrained(BASE_MT5)
    model=AutoModelForSeq2SeqLM.from_pretrained(BASE_MT5).to(DEVICE)
    freeze_encoder(model, num_layers=MT5_FREEZE_LAYERS)

    enc={}
    for sp,df in combined.items():
        enc[sp]=Dataset.from_pandas(df,preserve_index=False).map(preprocess_combined(tok),batched=True,remove_columns=list(df.columns))

    args=Seq2SeqTrainingArguments(
        output_dir=str(run_dir),learning_rate=1e-3,
        per_device_train_batch_size=TRAIN_BATCH,per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=EPOCHS,
        optim="adafactor",predict_with_generate=True,generation_max_length=MAX_LEN,
        fp16=False,eval_strategy="epoch",save_strategy="epoch",save_total_limit=CHECKPOINT_KEEP,
        load_best_model_at_end=True,metric_for_best_model="chrf",greater_is_better=True,
        logging_steps=25,report_to=["mlflow"],gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        disable_tqdm=True,
    )
    trainer=build_seq2seq_trainer(
        model=model,args=args,train_ds=enc["train"],eval_ds=enc["dev"],
        collator=DataCollatorForSeq2Seq(tok,model=model),
        compute_metrics=compute_metrics_builder(tok),tokenizer=tok,
        callbacks=[EpochArtifactCallback(run_dir), TqdmProgressCallback()]
    )
    from transformers.trainer_callback import PrinterCallback
    trainer.remove_callback(PrinterCallback)
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except ImportError:
        pass

    # Auto-detect this run's own last checkpoint -- same automatic resume as per-direction training.
    resume=RESUME_CHECKPOINT or get_last_checkpoint(str(run_dir))
    if resume:
        print(f"  [combined] resuming automatically from: {resume}")
    with mlflow.start_run(run_name="mt5-combined"):
        mlflow.log_params({"model":"mT5-small-combined","epochs":EPOCHS,
                           "freeze_layers":MT5_FREEZE_LAYERS,"learning_rate":1e-3,
                           "train_batch":TRAIN_BATCH,"grad_accum":GRAD_ACCUM,
                           "full_dataset_run":FULL_DATASET_RUN})
        t0=time.time()
        trainer.train(resume_from_checkpoint=resume)
        combined_runtime=(time.time()-t0)/60

        best=run_dir/"best"
        trainer.save_model(best); tok.save_pretrained(best)
        print("Combined training minutes:",round(combined_runtime,2))

        # Test evaluation, per direction -- done here, in the SAME function, so nothing depends
        # on an in-memory variable surviving a disconnect between separate cells.
        combined_rows=[]
        for slug in DIRECTIONS:
            src,tgt=slug.split("_to_")
            df=pd.read_csv(DATA_DIR/f"{slug}.test.csv").dropna(subset=["src_text","tgt_text"])
            preds=generate_mt5(trainer.model,tok,df["src_text"].tolist(),src,tgt)
            bleu=sacrebleu.corpus_bleu(preds,[df["tgt_text"].tolist()]).score
            chrf=sacrebleu.corpus_chrf(preds,[df["tgt_text"].tolist()],word_order=2).score
            combined_rows.append({"model":"mT5-small-combined","direction":slug,
                                  "setting":"combined","bleu":round(bleu,2),"chrf":round(chrf,2),
                                  "n_test":len(df),"runtime_min":combined_runtime,
                                  "full_dataset_run":FULL_DATASET_RUN})
        combined_df=pd.DataFrame(combined_rows)
        mlflow.log_metrics({"mean_bleu":combined_df["bleu"].mean(),"mean_chrf":combined_df["chrf"].mean(),
                            "runtime_min":combined_runtime})
    combined_df.to_csv(final_path,index=False)
    del model,trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return combined_df

combined_df=train_mt5_combined()
display(combined_df)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Map:   0%|          | 0/117080 [00:00<?, ? examples/s]

Map:   0%|          | 0/8781 [00:00<?, ? examples/s]

Map:   0%|          | 0/8784 [00:00<?, ? examples/s]

Training:   0%|          | 0/9150 [00:00<?, ?step/s]

...

  step  7325 | epoch  4.003 | loss  1.7569 | lr 2.00e-04 | grad_norm  0.646
  step  7350 | epoch  4.016 | loss  1.7940 | lr 1.97e-04 | grad_norm  0.626
  step  7375 | epoch  4.030 | loss  1.7608 | lr 1.94e-04 | grad_norm  0.716
  step  7400 | epoch  4.044 | loss  1.8547 | lr 1.91e-04 | grad_norm  0.554
  step  7425 | epoch  4.057 | loss  1.8334 | lr 1.89e-04 | grad_norm  0.509
  step  7450 | epoch  4.071 | loss  1.7952 | lr 1.86e-04 | grad_norm  0.545
  step  7475 | epoch  4.085 | loss  1.8923 | lr 1.83e-04 | grad_norm  1.013
  step  7500 | epoch  4.098 | loss  1.8603 | lr 1.80e-04 | grad_norm  0.568
  step  7525 | epoch  4.112 | loss  1.7978 | lr 1.78e-04 | grad_norm  0.634
  step  7550 | epoch  4.126 | loss  1.8409 | lr 1.75e-04 | grad_norm  0.670
  step  7575 | epoch  4.139 | loss  1.8469 | lr 1.72e-04 | grad_norm  1.185
  step  7600 | epoch  4.153 | loss  1.8220 | lr 1.70e-04 | grad_norm  0.602
  step  7625 | epoch  4.167 | loss  1.8410 | lr 1.67e-04 | grad_norm  1.167
  step  7650

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Combined training minutes: 169.67


,model,direction,setting,bleu,chrf,n_test,runtime_min,full_dataset_run
0,mT5-small-combined,English_to_Ekegusii,combined,15.42,38.01,2928,169.673468,True
1,mT5-small-combined,English_to_Kiswahili,combined,77.23,84.78,2928,169.673468,True
2,mT5-small-combined,Ekegusii_to_Kiswahili,combined,64.36,73.37,2928,169.673468,True


## Verify combined model actually generates real output

BLEU/chrF alone don't guarantee readable output -- this checks real translations directly,
since we found some checkpoints tonight that scored fine but produced garbage on generation.

In [229]:
# =========================================================================
# mT5 -- Full training + TRAIN/VALIDATION/TEST report, all in one cell
# =========================================================================

# ---- Explicitly override DIRECTIONS -- only these 2, matching NLLB's exact scope ----
DIRECTIONS = ["English_to_Ekegusii", "Kiswahili_to_Ekegusii"]

# ---- STEP 1: Train (skips anything already completed) ----
per_direction = []
for direction in DIRECTIONS:
    print(f"\n{'='*100}\nTraining: {direction}\n{'='*100}")
    per_direction.append(train_mt5_direction(direction=direction, freeze=FREEZE_ENCODER, augment=True, tag="per_direction"))

results_df = pd.DataFrame(per_direction)
results_df.to_csv(RESULTS_DIR / "mt5_selected_directions_final.csv", index=False)

# ---- STEP 2: TRAIN / VALIDATION / TEST report, per direction ----
print("\n" + "=" * 80)
print("MT5 -- FULL TRAIN / VALIDATION / TEST RESULTS")
print("=" * 80)

for direction in DIRECTIONS:
    run_dir = MT5_OUTPUT / "per_direction" / direction
    print(f"\n{'#'*80}\n# {direction}\n{'#'*80}")

    step_log = run_dir / "training_steps.csv"
    if step_log.exists():
        sdf = pd.read_csv(step_log)
        sdf["epoch_int"] = sdf["epoch"].apply(lambda x: int(x) + 1 if x % 1 != 0 else int(x))
        train_by_epoch = sdf.groupby("epoch_int")["loss"].mean().round(4)
        print("\n--- TRAIN (average loss per epoch) ---")
        for ep, loss in train_by_epoch.items():
            print(f"  Epoch {ep}: loss = {loss}")
    else:
        print("\n--- TRAIN --- \n  No step log found.")

    epoch_log = run_dir / "epoch_metrics.csv"
    if epoch_log.exists():
        edf = pd.read_csv(epoch_log)
        print("\n--- VALIDATION / DEV (one real evaluation per epoch) ---")
        for _, row in edf.iterrows():
            print(f"  Epoch {int(row['epoch'])}: loss = {row['eval_loss']:.4f}, "
                  f"bleu = {row['eval_bleu']:.2f}, chrf = {row['eval_chrf']:.2f}")
    else:
        print("\n--- VALIDATION / DEV --- \n  No epoch log found.")

    test_log = run_dir / "final_test_metrics.csv"
    if test_log.exists():
        tdf = pd.read_csv(test_log)
        print("\n--- TEST (final, held-out -- evaluated ONCE, not per epoch) ---")
        print(f"  bleu = {tdf['bleu'].iloc[0]:.2f}, chrf = {tdf['chrf'].iloc[0]:.2f}, "
              f"runtime_min = {tdf['runtime_min'].iloc[0]:.1f}")
    else:
        print("\n--- TEST --- \n  Not available.")

    best = run_dir / "best"
    if (best / "config.json").exists():
        src, tgt = direction.split("_to_")
        tok = AutoTokenizer.from_pretrained(best)
        model = AutoModelForSeq2SeqLM.from_pretrained(best)
        test_df = pd.read_csv(DATA_DIR / f"{direction}.test.csv").head(1)
        row = test_df.iloc[0]
        prompt = f"translate {src} to {tgt}: {row['src_text']}"
        enc = tok(prompt, return_tensors="pt", truncation=True, max_length=128)
        gen = model.generate(**enc, max_length=128)
        pred = tok.decode(gen[0], skip_special_tokens=True)
        print("\n--- REAL OUTPUT CHECK (1 example) ---")
        print(f"  SRC : {row['src_text']}")
        print(f"  PRED: {pred}")
        print(f"  REF : {row['tgt_text']}")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\n" + "=" * 80)


Training: English_to_Ekegusii


[W804 21:59:40.540235817 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2097152 bytes (free: 2621440, total: 85093777408).


Map:   0%|          | 0/46832 [00:00<?, ? examples/s]

Map:   0%|          | 0/2927 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

  [per_direction/English_to_Ekegusii] resuming automatically from: /home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/mt5/per_direction/English_to_Ekegusii/checkpoint-732

Training started


[W804 21:59:55.210937689 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 3426746368 bytes (free: 2202533888, total: 85093777408).
[W804 22:00:04.236869093 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 4097835008 bytes (free: 1208483840, total: 85093777408).


Epoch 2/5 completed | runtime 13.68 min
Epoch 3/5 completed | runtime 31.10 min
Epoch 4/5 completed | runtime 48.58 min
Epoch 5/5 completed | runtime 65.99 min
{'model': 'mT5-small', 'setting': 'per_direction', 'direction': 'English_to_Ekegusii', 'bleu': 13.72, 'chrf': 36.28, 'runtime_min': 69.68451350529989, 'epochs': 5, 'freeze_encoder': True, 'full_dataset_run': True}

Training: Kiswahili_to_Ekegusii
  [per_direction/Kiswahili_to_Ekegusii] already completed -- loading saved result, skipping retrain.

MT5 -- FULL TRAIN / VALIDATION / TEST RESULTS

################################################################################
# English_to_Ekegusii
################################################################################

--- TRAIN --- 
  No step log found.

--- VALIDATION / DEV (one real evaluation per epoch) ---
  Epoch 1: loss = 2.1188, bleu = 6.08, chrf = 26.23
  Epoch 2: loss = 1.8276, bleu = 10.47, chrf = 32.38
  Epoch 3: loss = 1.7046, bleu = 12.90, chrf = 35.46
  Epoch

In [186]:
def train_mt5_direction(direction,freeze=True,augment=True,tag="per_direction"):
    run_dir=MT5_OUTPUT/tag/direction
    run_dir.mkdir(parents=True,exist_ok=True)

    final_path=run_dir/"final_test_metrics.csv"
    if final_path.exists():
        print(f"  [{tag}/{direction}] already completed -- loading saved result, skipping retrain.")
        return pd.read_csv(final_path).iloc[0].to_dict()

    src,tgt=direction.split("_to_")
    dsd=load_direction(direction,augment=augment)
    tok=AutoTokenizer.from_pretrained(BASE_MT5)
    model=AutoModelForSeq2SeqLM.from_pretrained(BASE_MT5).to(DEVICE)
    if freeze: freeze_encoder(model, num_layers=MT5_FREEZE_LAYERS)
    enc=dsd.map(preprocess_mt5(tok,src,tgt),batched=True,remove_columns=dsd["train"].column_names)

    args=Seq2SeqTrainingArguments(
        output_dir=str(run_dir),learning_rate=1e-3,
        per_device_train_batch_size=TRAIN_BATCH,per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=EPOCHS,
        optim="adafactor",weight_decay=0.0,predict_with_generate=True,
        generation_max_length=MAX_LEN,fp16=False,
        eval_strategy="epoch",save_strategy="epoch",save_total_limit=CHECKPOINT_KEEP,
        load_best_model_at_end=True,metric_for_best_model="chrf",greater_is_better=True,
        logging_steps=25,report_to=["mlflow"],gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
        disable_tqdm=True,
    )
    trainer=build_seq2seq_trainer(
        model=model,args=args,train_ds=enc["train"],eval_ds=enc["dev"],
        collator=DataCollatorForSeq2Seq(tok,model=model),
        compute_metrics=compute_metrics_builder(tok),tokenizer=tok,
        callbacks=[EpochArtifactCallback(run_dir), TqdmProgressCallback()]
    )
    from transformers.trainer_callback import PrinterCallback
    trainer.remove_callback(PrinterCallback)
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except ImportError:
        pass

    resume=RESUME_CHECKPOINT or get_last_checkpoint(str(run_dir))
    if resume:
        print(f"  [{tag}/{direction}] resuming automatically from: {resume}")
    with mlflow.start_run(run_name=f"mt5-{tag}-{direction}"):
        mlflow.log_params({"model":"mT5-small","direction":direction,"tag":tag,"epochs":EPOCHS,
                           "freeze_encoder":freeze,"freeze_layers":MT5_FREEZE_LAYERS if freeze else 0,
                           "learning_rate":1e-3,"train_batch":TRAIN_BATCH,"grad_accum":GRAD_ACCUM,
                           "full_dataset_run":FULL_DATASET_RUN})
        t0=time.time()
        trainer.train(resume_from_checkpoint=resume)
        runtime=(time.time()-t0)/60

        best=run_dir/"best"
        trainer.save_model(best); tok.save_pretrained(best)

        test=enc["test"]
        ev=trainer.predict(test,metric_key_prefix="test")
        final={"model":"mT5-small","setting":tag,"direction":direction,
               "bleu":ev.metrics.get("test_bleu"),"chrf":ev.metrics.get("test_chrf"),
               "runtime_min":runtime,"epochs":EPOCHS,"freeze_encoder":freeze,
               "full_dataset_run":FULL_DATASET_RUN}
        mlflow.log_metrics({k:v for k,v in final.items() if isinstance(v,(int,float))})

        preds_ids = ev.predictions[0] if isinstance(ev.predictions, tuple) else ev.predictions
        preds_ids = np.where(preds_ids != -100, preds_ids, tok.pad_token_id)
        labels = np.where(ev.label_ids != -100, ev.label_ids, tok.pad_token_id)
        decoded_preds = tok.batch_decode(preds_ids, skip_special_tokens=True)
        decoded_refs = tok.batch_decode(labels, skip_special_tokens=True)
        sample_df = pd.DataFrame({
            "source": dsd["test"]["src_text"][:10],
            "prediction": decoded_preds[:10],
            "reference": decoded_refs[:10],
        })
        sample_df.to_csv(run_dir / "sample_translations.csv", index=False)

    pd.DataFrame([final]).to_csv(final_path,index=False)
    print(final)
    del model,trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final

In [202]:
from transformers import TrainerCallback
import pandas as pd
import time
from pathlib import Path
from tqdm.auto import tqdm


class EpochArtifactCallback(TrainerCallback):
    """
    Saves epoch-level validation metrics and training progress.
    """

    def __init__(self, run_dir):
        self.run_dir = Path(run_dir)
        self.run_dir.mkdir(parents=True, exist_ok=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):

        if metrics is None:
            return

        row = {
            "epoch": state.epoch,
            "eval_loss": metrics.get("eval_loss"),
            "eval_bleu": metrics.get("eval_bleu"),
            "eval_chrf": metrics.get("eval_chrf"),
        }

        path = self.run_dir / "epoch_metrics.csv"

        df = pd.DataFrame([row])

        if path.exists():
            old = pd.read_csv(path)
            df = pd.concat([old, df], ignore_index=True)

        df.to_csv(path, index=False)



class TqdmProgressCallback(TrainerCallback):
    """
    Simple console progress callback.
    """

    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print("\nTraining started")

    def on_epoch_end(self, args, state, control, **kwargs):
        elapsed = (time.time() - self.start_time) / 60
        print(
            f"Epoch {int(state.epoch)}/{int(args.num_train_epochs)} completed "
            f"| runtime {elapsed:.2f} min"
        )


print("Callbacks restored")
print(EpochArtifactCallback)
print(TqdmProgressCallback)

Callbacks restored
<class '__main__.EpochArtifactCallback'>
<class '__main__.TqdmProgressCallback'>


## 11. Combined model: held-out test evaluation by direction

In [24]:
# Test evaluation now happens inside train_mt5_combined() itself (cell above) --
# this just re-displays the saved result, safe to re-run any time with no side effects.
combined_df = pd.read_csv(RESULTS_DIR/"mt5_combined_final.csv")
display(combined_df)

,model,direction,setting,bleu,chrf,n_test,runtime_min,full_dataset_run
0,mT5-small-combined,English_to_Ekegusii,combined,15.42,38.01,2928,169.673468,True
1,mT5-small-combined,English_to_Kiswahili,combined,77.23,84.78,2928,169.673468,True
2,mT5-small-combined,Ekegusii_to_Kiswahili,combined,64.36,73.37,2928,169.673468,True


## 12. Transfer-learning ablation: combined vs per-direction

In [27]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path("/home/jovyan/PSA_MT_Project/PSA-MT-Outputs/results")

per = pd.read_csv(RESULTS_DIR/"mt5_selected_directions_final.csv")
comb = pd.read_csv(RESULTS_DIR/"mt5_combined_final.csv")

cmp = per.merge(comb, on="direction", suffixes=("_per_direction", "_combined"))

print("="*100)
print("MT5 FINAL COMPARISON")
print("="*100)

display(cmp)

print("\nShapes:")
print(f"Per-direction: {per.shape} | Combined: {comb.shape} | Comparison: {cmp.shape}")

MT5 FINAL COMPARISON


,model_per_direction,setting_per_direction,direction,bleu_per_direction,chrf_per_direction,runtime_min_per_direction,epochs,freeze_encoder,full_dataset_run_per_direction,model_combined,setting_combined,bleu_combined,chrf_combined,n_test,runtime_min_combined,full_dataset_run_combined
0,mT5-small,per_direction,English_to_Ekegusii,13.850536,36.364417,56.126775,5,True,True,mT5-small-combined,combined,15.42,38.01,2928,169.673468,True
1,mT5-small,per_direction,English_to_Kiswahili,77.990718,85.780823,0.026502,5,True,True,mT5-small-combined,combined,77.23,84.78,2928,169.673468,True
2,mT5-small,per_direction,Ekegusii_to_Kiswahili,64.332173,73.432127,39.891198,5,True,True,mT5-small-combined,combined,64.36,73.37,2928,169.673468,True



Shapes:
Per-direction: (3, 9) | Combined: (3, 8) | Comparison: (3, 16)


In [29]:
per = pd.read_csv(RESULTS_DIR/"mt5_selected_directions_final.csv")
comb = pd.read_csv(RESULTS_DIR/"mt5_combined_final.csv")
cmp = per.merge(comb, on="direction", how="left", suffixes=("_perdir", "_combined"))
cmp["chrf_gain_combined"] = cmp["chrf_combined"] - cmp["chrf_perdir"]
cmp["bleu_gain_combined"] = cmp["bleu_combined"] - cmp["bleu_perdir"]

print("="*100)
print("MT5 FINAL COMPARISON (4 directions -- Kiswahili_to_Ekegusii has no 'combined' result, shown as NaN)")
print("="*100)
display(cmp[["direction","bleu_perdir","bleu_combined","bleu_gain_combined",
            "chrf_perdir","chrf_combined","chrf_gain_combined"]])
cmp.to_csv(RESULTS_DIR/"combined_vs_perdirection.csv", index=False)

,direction,bleu_perdir,bleu_combined,bleu_gain_combined,chrf_perdir,chrf_combined,chrf_gain_combined
0,English_to_Ekegusii,13.850536,15.42,1.569464,36.364417,38.01,1.645583
1,English_to_Kiswahili,77.990718,77.23,-0.760718,85.780823,84.78,-1.000823
2,Ekegusii_to_Kiswahili,64.332173,64.36,0.027827,73.432127,73.37,-0.062127


## 13. Freeze vs full fine-tuning ablation

Run this on **one representative direction** to isolate the effect of encoder freezing. The output is stored separately so it does not overwrite the main per-direction baseline.

In [30]:
ABLATION_DIR="English_to_Ekegusii"
full_ft=train_mt5_direction(ABLATION_DIR,freeze=False,augment=False,tag="ablation_full_finetune")
pd.DataFrame([full_ft]).to_csv(RESULTS_DIR/"freeze_vs_full_ablation.csv",index=False)
print("Compare this with the corresponding frozen model in mt5_per_direction_final.csv")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Map:   0%|          | 0/23416 [00:00<?, ? examples/s]

Map:   0%|          | 0/2927 [00:00<?, ? examples/s]

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

Training:   0%|          | 0/1830 [00:00<?, ?step/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  step  1475 | epoch  4.030 | loss  3.8955 | lr 1.95e-04 | grad_norm  1.106
  step  1500 | epoch  4.098 | loss  3.7674 | lr 1.81e-04 | grad_norm  1.114
  step  1525 | epoch  4.167 | loss  3.8618 | lr 1.67e-04 | grad_norm  0.964
  step  1550 | epoch  4.235 | loss  3.8618 | lr 1.54e-04 | grad_norm  0.940
  step  1575 | epoch  4.303 | loss  3.8571 | lr 1.40e-04 | grad_norm  0.934
  step  1600 | epoch  4.372 | loss  3.8738 | lr 1.26e-04 | grad_norm  0.925
  step  1625 | epoch  4.440 | loss  3.8262 | lr 1.13e-04 | grad_norm  0.919
  step  1650 | epoch  4.508 | loss  3.8583 | lr 9.89e-05 | grad_norm  1.204
  step  1675 | epoch  4.577 | loss  3.8644 | lr 8.52e-05 | grad_norm  1.175
  step  1700 | epoch  4.645 | loss  3.7374 | lr 7.16e-05 | grad_norm  1.086
  step  1725 | epoch  4.713 | loss  3.7301 | lr 5.79e-05 | grad_norm  1.007
  step  1750 | epoch  4.781 | loss  3.7691 | lr 4.43e-05 | grad_norm  0.963
  step  1775 | epoch  4.850 | loss  3.9239 | lr 3.06e-05 | grad_norm  1.024
  step  1800

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'model': 'mT5-small', 'setting': 'ablation_full_finetune', 'direction': 'English_to_Ekegusii', 'bleu': 12.929189339981358, 'chrf': 34.990920208406266, 'runtime_min': 33.634473824501036, 'epochs': 5, 'freeze_encoder': False, 'full_dataset_run': True}
Compare this with the corresponding frozen model in mt5_per_direction_final.csv


## 14. Ablation study — domain adaptation

Breaks down the fine-tuned **combined** mT5 model's performance by PSA domain (Education,
Health, Agriculture, Governance, Security). Domains with fewer than 5 test examples are
skipped -- too few to score meaningfully. This is a genuine ablation, not just descriptive:
it tells you whether the model generalizes evenly across domains or has specific weak spots
(e.g. technical/legal vocabulary in Governance vs. simpler everyday phrasing in Health).

### **Domain adaptation, per-direction**

In [70]:
from tqdm.auto import tqdm
import torch

@torch.no_grad()
def generate_mt5(model,tokenizer,texts,src,tgt,batch_size=8,max_len=128):
    model.eval(); predictions=[]
    for i in tqdm(range(0,len(texts),batch_size),desc=f"{src}->{tgt}"):
        batch=texts[i:i+batch_size]
        prompts=[f"translate {src} to {tgt}: {x}" for x in batch]
        enc=tokenizer(prompts,return_tensors="pt",padding=True,truncation=True,max_length=max_len).to(model.device)
        outputs=model.generate(**enc,max_length=max_len,num_beams=4)
        predictions.extend(tokenizer.batch_decode(outputs,skip_special_tokens=True))
    return predictions

print("generate_mt5 ready")

generate_mt5 ready


In [24]:
DIRECTIONS = ["English_to_Ekegusii", "Kiswahili_to_Ekegusii", "English_to_Kiswahili"]

(RESULTS_DIR/"mt5_domain_adaptation_per_direction.csv").unlink(missing_ok=True)

def domain_adaptation_ablation_per_direction(min_examples=5):
    out_path = RESULTS_DIR/"mt5_domain_adaptation_per_direction.csv"
    if out_path.exists():
        print("  per-direction domain-adaptation ablation already computed -- loading saved result.")
        return pd.read_csv(out_path)

    rows = []
    for slug in DIRECTIONS:
        model_dir = MT5_OUTPUT/"per_direction"/slug/"best"
        if not (model_dir/"config.json").exists():
            print(f"  {slug}: no per-direction checkpoint yet -- skipping."); continue
        src, tgt = slug.split("_to_")
        tok = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(DEVICE)
        df = pd.read_csv(DATA_DIR/f"{slug}.test.csv").dropna(subset=["src_text","tgt_text"])
        if "Domain" not in df.columns:
            print("  no Domain column -- skipping."); del model; continue
        for dom in df["Domain"].unique():
            sub = df[df["Domain"]==dom]
            if len(sub) < min_examples: continue
            preds = generate_mt5(model, tok, sub["src_text"].tolist(), src, tgt)
            bleu = sacrebleu.corpus_bleu(preds,[sub["tgt_text"].tolist()]).score
            chrf = sacrebleu.corpus_chrf(preds,[sub["tgt_text"].tolist()],word_order=2).score
            rows.append({"direction":slug,"domain":dom,"bleu":round(bleu,2),"chrf":round(chrf,2),"n_examples":len(sub)})
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    domain_df = pd.DataFrame(rows)
    domain_df.to_csv(out_path, index=False)
    return domain_df

domain_df_per_dir = domain_adaptation_ablation_per_direction()
display(domain_df_per_dir)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

English->Ekegusii:   0%|          | 0/50 [00:00<?, ?it/s]

English->Ekegusii:   0%|          | 0/176 [00:00<?, ?it/s]

English->Ekegusii:   0%|          | 0/50 [00:00<?, ?it/s]

English->Ekegusii:   0%|          | 0/49 [00:00<?, ?it/s]

...

English->Kiswahili:   0%|          | 0/176 [00:00<?, ?it/s]

English->Kiswahili:   0%|          | 0/50 [00:00<?, ?it/s]

English->Kiswahili:   0%|          | 0/49 [00:00<?, ?it/s]

English->Kiswahili:   0%|          | 0/44 [00:00<?, ?it/s]

,direction,domain,bleu,chrf,n_examples
0,English_to_Ekegusii,Health,19.75,42.84,396
1,English_to_Ekegusii,Agriculture,11.58,33.75,1401
2,English_to_Ekegusii,Education,20.78,44.62,394
3,English_to_Ekegusii,Security & Safety,20.36,40.72,388
4,English_to_Ekegusii,Governance,16.07,39.68,349
5,Kiswahili_to_Ekegusii,Health,20.46,42.56,396
6,Kiswahili_to_Ekegusii,Agriculture,12.75,34.99,1401
7,Kiswahili_to_Ekegusii,Education,19.44,42.59,394
8,Kiswahili_to_Ekegusii,Security & Safety,19.03,39.52,388
9,Kiswahili_to_Ekegusii,Governance,15.83,38.24,349


### **Domain adaptation, per-direction**

In [31]:
def domain_adaptation_ablation(model_dir=MT5_OUTPUT/"combined"/"best", min_examples=5):
    out_path = RESULTS_DIR/"mt5_domain_adaptation.csv"
    if out_path.exists():
        print("  domain-adaptation ablation already computed -- loading saved result.")
        return pd.read_csv(out_path)
    if not (model_dir/"config.json").exists():
        print(f"  {model_dir} not trained yet -- run the combined-model cell first.")
        return None

    tok=AutoTokenizer.from_pretrained(model_dir)
    model=AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(DEVICE)
    rows=[]
    for slug in DIRECTIONS:
        src,tgt=slug.split("_to_")
        df=pd.read_csv(DATA_DIR/f"{slug}.test.csv").dropna(subset=["src_text","tgt_text"])
        if "Domain" not in df.columns:
            print("  no Domain column in test data -- skipping domain ablation."); return None
        for dom in df["Domain"].unique():
            sub=df[df["Domain"]==dom]
            if len(sub) < min_examples:
                continue
            preds=generate_mt5(model,tok,sub["src_text"].tolist(),src,tgt)
            bleu=sacrebleu.corpus_bleu(preds,[sub["tgt_text"].tolist()]).score
            chrf=sacrebleu.corpus_chrf(preds,[sub["tgt_text"].tolist()],word_order=2).score
            rows.append({"direction":slug,"domain":dom,"bleu":round(bleu,2),
                        "chrf":round(chrf,2),"n_examples":len(sub)})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    domain_df=pd.DataFrame(rows)
    domain_df.to_csv(out_path,index=False)
    return domain_df

domain_df = domain_adaptation_ablation()
if domain_df is not None:
    display(domain_df)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

,direction,domain,bleu,chrf,n_examples
0,English_to_Ekegusii,Health,18.98,43.42,396
1,English_to_Ekegusii,Agriculture,10.69,33.60,1401
2,English_to_Ekegusii,Education,20.28,43.49,394
3,English_to_Ekegusii,Security & Safety,19.19,39.28,388
4,English_to_Ekegusii,Governance,17.34,39.66,349
5,English_to_Kiswahili,Health,78.26,84.76,396
6,English_to_Kiswahili,Agriculture,76.30,84.48,1401
7,English_to_Kiswahili,Education,82.31,88.15,394
8,English_to_Kiswahili,Security & Safety,73.28,81.48,388
9,English_to_Kiswahili,Governance,79.39,86.43,349


## 15. COMET on final mT5 models

COMET is calculated **after training**, not every epoch, to keep training time manageable.

## COMET install (isolated, run only when you reach this point)

`unbabel-comet` depends on an old `pytorch-lightning` pin with malformed metadata that recent
`pip` rejects. Installing it in its own cell -- separate from the core packages -- means this
can never block training. If it fails again, tell me the exact error before retrying.

In [1]:
import comet
import transformers
import torch

print("COMET:", comet.__version__)
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)


/opt/conda/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


COMET: 2.2.7
Transformers: 4.57.6
Torch: 2.13.0+cu130


In [2]:
from comet import download_model, load_from_checkpoint

print("Downloading COMET model...")
comet_model_path = download_model("Unbabel/wmt22-comet-da")

print("Loading COMET checkpoint...")
comet_model = load_from_checkpoint(comet_model_path)

print("COMET model loaded successfully")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading COMET checkpoint...


Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.


COMET model loaded successfully


/opt/conda/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [23]:
# from tqdm.auto import tqdm
# import torch

# @torch.no_grad()
# def generate_mt5(model,tokenizer,texts,src,tgt,batch_size=8,max_len=128):
#     model.eval(); predictions=[]
#     for i in tqdm(range(0,len(texts),batch_size),desc=f"{src}->{tgt}"):
#         batch=texts[i:i+batch_size]
#         prompts=[f"translate {src} to {tgt}: {x}" for x in batch]
#         enc=tokenizer(prompts,return_tensors="pt",padding=True,truncation=True,max_length=max_len).to(model.device)
#         outputs=model.generate(**enc,max_length=max_len,num_beams=4)
#         predictions.extend(tokenizer.batch_decode(outputs,skip_special_tokens=True))
#     return predictions

# print("generate_mt5 ready")

generate_mt5 ready


In [73]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained(
#     "google/mt5-small",
#     use_fast=False
# )

# print("Tokenizer loaded successfully")

Tokenizer loaded successfully


In [218]:
# DIRECTIONS = ["English_to_Ekegusii", "Kiswahili_to_Ekegusii", "English_to_Kiswahili"]

# comet_out_path = RESULTS_DIR / "mt5_perdirection_comet.csv"

# if comet_out_path.exists():
#     print("Per-direction COMET already computed -- loading saved result.")
#     comet_perdir_df = pd.read_csv(comet_out_path)
# else:
#     comet_perdir_rows = []
#     for slug in DIRECTIONS:
#         print(f"\nProcessing: {slug}")
#         model_dir = MT5_OUTPUT / "per_direction" / slug / "best"
#         if not (model_dir / "config.json").exists():
#             print(f"  {slug}: no checkpoint yet -- skipping.")
#             continue

#         src, tgt = slug.split("_to_")
#         tok = AutoTokenizer.from_pretrained(model_dir)
#         model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(DEVICE).eval()

#         df = pd.read_csv(DATA_DIR / f"{slug}.test.csv").dropna(subset=["src_text", "tgt_text"])
#         preds = generate_mt5(model, tok, df["src_text"].tolist(), src, tgt)

#         comet_data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(df.src_text, preds, df.tgt_text)]
#         out = comet_model.predict(comet_data, batch_size=8, gpus=0, num_workers=0)
#         score = round(float(out.system_score) * 100, 2)
#         print(f"{slug}: COMET = {score}")

#         comet_perdir_rows.append({"model": "mT5-small-per-direction", "direction": slug,
#                                   "comet": score, "samples": len(df)})
#         del model
#         torch.cuda.empty_cache()

#     comet_perdir_df = pd.DataFrame(comet_perdir_rows)
#     comet_perdir_df.to_csv(comet_out_path, index=False)

# display(comet_perdir_df)

In [222]:
from pathlib import Path

MODEL_DIR = Path(
"/home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/mt5/ablation_full_finetune/English_to_Ekegusii/best"
)

print(MODEL_DIR.exists())

for f in MODEL_DIR.iterdir():
    print(f.name)

True
spiece.model
tokenizer.json
generation_config.json
special_tokens_map.json
model.safetensors
training_args.bin
tokenizer_config.json
config.json


## 16. Demo — quick inference on sample PSAs

Satisfies "Week 3 deliverable: working translation demo." Uses the combined model (all 6
directions in one checkpoint) if trained, falling back to a per-direction checkpoint
otherwise. This is also what gets extracted into the standalone `translate_psa.py` CLI
script below, so both stay in sync.

In [230]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_DIR = "/home/jovyan/PSA_MT_Project/PSA-MT-Outputs/models/fine_tuned/mt5/ablation_full_finetune/English_to_Ekegusii/best"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*80)
print("mT5 ENGLISH → EKEGUSII TEST")
print("="*80)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
).to(DEVICE)

model.eval()

source = (
    "Division of Disease Surveillance and Response calls on parents "
    "in Isiolo County to have children under five screened for HIV/AIDS "
    "during the June outreach."
)

prompt = f"translate English to Ekegusii: {source}"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128
).to(DEVICE)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_length=128,
        num_beams=4,
        repetition_penalty=1.2,
        early_stopping=True
    )

prediction = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("\nSOURCE:")
print(source)

print("\nPREDICTION:")
print(prediction)

del model
torch.cuda.empty_cache()

mT5 ENGLISH → EKEGUSII TEST

SOURCE:
Division of Disease Surveillance and Response calls on parents in Isiolo County to have children under five screened for HIV/AIDS during the June outreach.

PREDICTION:
MALLGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHzGHz


## 18. Failure recovery -- now fully automatic

**If a session disconnects at any point (mid-epoch, between directions, overnight, whenever):
just re-run the notebook from the top. No manual steps needed.**

What happens automatically on re-run:
- Any direction that already finished (`final_test_metrics.csv` exists) is **skipped entirely** --
  zero wasted GPU time, no risk of accidentally overwriting a good result.
- Any direction that was mid-training is **automatically resumed from its own last epoch
  checkpoint** (`get_last_checkpoint`) -- you do NOT need to find or set `RESUME_CHECKPOINT`
  by hand anymore. That variable is still there only as a manual override if you ever want to
  force a specific checkpoint.
- This applies to per-direction training, the combined model, and the freeze-vs-full ablation --
  all three go through the same skip/resume logic.

For a genuine hosted-notebook session **reset** (not just a disconnect): remount Drive and re-run the
setup/data cells (1-9) before re-running training -- your checkpoints and completed results are
untouched on Drive either way.

**This is what makes an unattended multi-day run (e.g. over a weekend) safe**: if it disconnects at
2am, whoever reconnects it in the morning just has to click "Run all" again.

## 19. Final artifact checklist

Before moving to deployment/demo, confirm:

- [ ] shared split manifest exists
- [ ] all six train/dev/test files exist
- [ ] base mT5 checkpoint is saved
- [ ] every completed epoch has a checkpoint
- [ ] `epoch_metrics.csv` exists for each run
- [ ] best combined model exists
- [ ] BLEU, SacreBLEU/chrF++, and COMET results exist
- [ ] combined-vs-per-direction ablation exists
- [ ] freeze-vs-full-finetune ablation exists
- [ ] results are copied into the final report

In [227]:
from pathlib import Path

ROOT = Path("/home/jovyan/PSA_MT_Project/PSA-MT-Outputs-v2/models/fine_tuned")

print("="*120)
print("PSA-MT-Outputs-v2")
print("="*120)

for model in sorted(ROOT.rglob("best")):

    print("\nMODEL")
    print(model)

    for f in sorted(model.iterdir()):
        print("   ", f.name)

PSA-MT-Outputs-v2

MODEL
/home/jovyan/PSA_MT_Project/PSA-MT-Outputs-v2/models/fine_tuned/nllb/English_to_Ekegusii/best
    config.json
    generation_config.json
    model.safetensors
    tokenizer.json
    tokenizer_config.json
    training_args.bin

MODEL
/home/jovyan/PSA_MT_Project/PSA-MT-Outputs-v2/models/fine_tuned/nllb/Kiswahili_to_Ekegusii/best
    config.json
    generation_config.json
    model.safetensors
    tokenizer.json
    tokenizer_config.json
    training_args.bin
